# Research → Analysis → Report Pipeline

A modular multi-agent workflow using the **OpenAI Agents SDK**.  
The `Researcher` agent is isolated so it can be swapped for any domain-specific researcher without touching the rest of the pipeline.

## 1. Setup — Install Dependencies

In [ ]:
%pip install -q openai-agents python-dotenv pydantic requests

## 2. Imports & Environment

In [ ]:
import os
import asyncio
import requests
from IPython.display import Markdown, display
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List

from agents import Agent, Runner, function_tool

load_dotenv()

# Validate required keys are present
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not found in environment"

print("Environment loaded successfully.")

## 3. Pydantic Output Models

In [ ]:
class AnalystInsights(BaseModel):
    key_trends: List[str] = Field(description="Major trends identified in the research")
    risks: List[str] = Field(description="Key risks or challenges identified")
    insights: List[str] = Field(description="Actionable insights and observations")


class FinalReport(BaseModel):
    executive_summary: str = Field(description="A concise executive summary (2-4 sentences)")
    markdown_report: str = Field(description="A full markdown-formatted report covering findings, analysis, and recommendations")
    follow_up_questions: List[str] = Field(description="3-5 follow-up questions for further investigation")

## 4. Tavily Search Tool

In [ ]:
@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """
    Search the web using Tavily to retrieve relevant, up-to-date information.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return (default 5).

    Returns:
        A formatted string containing search results with titles, URLs, and content snippets.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    payload = {
        "api_key": api_key,
        "query": query,
        "max_results": max_results,
    }
    response = requests.post("https://api.tavily.com/search", json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    results = data.get("results", [])
    if not results:
        return "No results found."

    formatted = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "No title")
        url = r.get("url", "")
        content = r.get("content", "")
        formatted.append(f"[{i}] **{title}**\nURL: {url}\n{content}\n")

    return "\n".join(formatted)

## 5. Agent Definitions

### 5a. Researcher Agent Factory

The `build_researcher_agent()` function is the **only thing that needs to change** when swapping the Researcher for a different domain.

In [ ]:
def build_researcher_agent() -> Agent:
    """
    Factory function that creates and returns the Researcher agent.
    Swap this function out to replace the Researcher for a different domain.
    """
    return Agent(
        name="Researcher",
        instructions=(
            "You are an expert research assistant. When given a topic or question, "
            "you MUST use the tavily_search tool to gather information from the web before summarizing. "
            "Perform multiple targeted searches if needed to get comprehensive coverage. "
            "Synthesize the search results into a well-organized research summary that includes: "
            "key facts, recent developments, relevant statistics, and cited sources (URLs). "
            "Be thorough and objective."
        ),
        tools=[tavily_search],
        model="gpt-4o-mini",
    )

### 5b. Analyst Agent

In [ ]:
analyst_agent = Agent(
    name="Analyst",
    instructions=(
        "You are a senior analyst. You will receive a research summary and must extract: "
        "(1) Key trends shaping the topic, "
        "(2) Significant risks or challenges, "
        "(3) Actionable insights and strategic observations. "
        "Be specific, evidence-based, and concise. Return your output as structured JSON."
    ),
    output_type=AnalystInsights,
    model="gpt-4o-mini",
)

### 5c. Writer Agent

In [ ]:
writer_agent = Agent(
    name="Writer",
    instructions=(
        "You are a professional technical writer. You will receive a research summary and analyst insights. "
        "Produce a polished deliverable consisting of: "
        "(1) A concise executive summary (2-4 sentences suitable for a busy executive), "
        "(2) A full markdown-formatted report with sections: Overview, Key Findings, Trends & Risks, "
        "Recommendations, and Sources, "
        "(3) 3-5 thoughtful follow-up questions for further investigation. "
        "Return your output as structured JSON."
    ),
    output_type=FinalReport,
    model="gpt-4o-mini",
)

## 6. Pipeline Orchestrator

In [ ]:
async def run_pipeline(user_query: str, researcher_agent: Agent) -> FinalReport:
    """
    Orchestrates the Researcher → Analyst → Writer pipeline.

    Args:
        user_query: The research question or topic.
        researcher_agent: An Agent instance for the Researcher step.
                          Pass a different agent here to swap out the Researcher.

    Returns:
        A FinalReport Pydantic model with executive_summary, markdown_report,
        and follow_up_questions.
    """
    print("[1/3] Researcher gathering sources...")
    research_result = await Runner.run(researcher_agent, user_query)
    research_summary = research_result.final_output

    print("[2/3] Analyst extracting insights...")
    analyst_input = (
        f"Research Summary:\n{research_summary}\n\n"
        f"Original Query: {user_query}"
    )
    analyst_result = await Runner.run(analyst_agent, analyst_input)
    insights: AnalystInsights = analyst_result.final_output

    print("[3/3] Writer composing final report...")
    writer_input = (
        f"Original Query: {user_query}\n\n"
        f"Research Summary:\n{research_summary}\n\n"
        f"Analyst Insights:\n"
        f"  Key Trends: {insights.key_trends}\n"
        f"  Risks: {insights.risks}\n"
        f"  Insights: {insights.insights}"
    )
    writer_result = await Runner.run(writer_agent, writer_input)
    final_report: FinalReport = writer_result.final_output

    print("Pipeline complete.")
    return final_report

## 7. Output Formatter

In [ ]:
def display_report(report: FinalReport) -> None:
    """Render the final report cleanly in the notebook."""
    display(Markdown("---"))
    display(Markdown("## Executive Summary"))
    display(Markdown(report.executive_summary))

    display(Markdown("---"))
    display(Markdown("## Full Report"))
    display(Markdown(report.markdown_report))

    display(Markdown("---"))
    display(Markdown("## Follow-Up Questions"))
    questions_md = "\n".join(f"{i}. {q}" for i, q in enumerate(report.follow_up_questions, 1))
    display(Markdown(questions_md))

## 8. Example Run

Run the full pipeline with a sample query.  
To swap the Researcher, replace `build_researcher_agent()` with your own factory function.

In [ ]:
SAMPLE_QUERY = "What are the latest developments and key trends in AI agents and agentic frameworks in 2025?"

# Build the researcher (swap this for a different domain by calling a different factory)
researcher = build_researcher_agent()

# Run the pipeline
report = await run_pipeline(SAMPLE_QUERY, researcher)

# Display the output
display_report(report)